# 07 — Ragas Evaluation
**Phase:** Evaluation & CI — Day 10

**What this notebook does:**
- Creates a golden dataset of 20 question-answer pairs from your PDF
- Runs Ragas evaluation metrics: faithfulness, answer relevancy, context precision
- Saves results to `data/ragas_results.json`
- Produces a score you can quote in your resume and LinkedIn post

**What is Ragas?**
Ragas measures RAG quality without needing human judges:
- **Faithfulness**: are all claims in the answer supported by retrieved chunks?
- **Answer Relevancy**: does the answer actually address the question?
- **Context Precision**: were the right chunks retrieved (not irrelevant ones)?

In [ ]:
!pip install ragas datasets -q

In [ ]:
import chromadb
import numpy as np
import json
import os
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder
from langchain_groq import ChatGroq
from langchain.schema import HumanMessage, SystemMessage
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from dotenv import load_dotenv

load_dotenv()
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection("my_docs")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
llm = ChatGroq(model="llama3-8b-8192", temperature=0.0, api_key=GROQ_API_KEY)

all_data   = collection.get(include=["documents"])
all_chunks = all_data["documents"]
all_ids    = all_data["ids"]
tokenised  = [c.lower().split() for c in all_chunks]
bm25       = BM25Okapi(tokenised)

print("All components loaded")

## Step 1 — Rebuild the production pipeline

In [ ]:
def vector_search(query, top_k=10):
    vec = embed_model.encode([query]).tolist()
    res = collection.query(query_embeddings=vec, n_results=top_k)
    return res["ids"][0], res["documents"][0]

def bm25_search(query, top_k=10):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return [all_ids[i] for i in top_idx], [all_chunks[i] for i in top_idx]

def rrf_fusion(v_ids, b_ids, k=60):
    scores = {}
    for rank, cid in enumerate(v_ids):
        scores[cid] = scores.get(cid, 0) + 1/(k+rank+1)
    for rank, cid in enumerate(b_ids):
        scores[cid] = scores.get(cid, 0) + 1/(k+rank+1)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)

SYSTEM_PROMPT = """You are DocMind, a precise document assistant.
Answer using ONLY the provided context. Be concise.
End with: Sources: [chunk_id]
If insufficient context, say: INSUFFICIENT_CONTEXT"""

def full_rag_pipeline(query):
    """Returns answer and retrieved contexts for Ragas evaluation."""
    v_ids, v_chunks = vector_search(query)
    b_ids, b_chunks = bm25_search(query)
    fused = rrf_fusion(v_ids, b_ids)
    id_to_chunk = {cid: c for cid, c in zip(v_ids+b_ids, v_chunks+b_chunks)}
    candidates = [{"id": cid, "text": id_to_chunk[cid]} for cid, _ in fused[:10] if cid in id_to_chunk]
    pairs = [(query, c["text"]) for c in candidates]
    rerank_scores = reranker.predict(pairs)
    for i, c in enumerate(candidates):
        c["rerank_score"] = float(rerank_scores[i])
    top3 = sorted(candidates, key=lambda x: x["rerank_score"], reverse=True)[:3]

    context_str = "".join([f"[{r['id']}]\n{r['text']}\n" for r in top3])
    messages = [
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=f"Context:\n{context_str}\n---\nQuestion: {query}\nAnswer:")
    ]
    response = llm.invoke(messages)
    return response.content, [r["text"] for r in top3]

print("Pipeline ready")

## Step 2 — Create your golden dataset

**IMPORTANT:** Replace these with 20 real questions from YOUR PDF.
The ground truth answers should be factually correct based on your document.
Write at least 20 pairs for a meaningful score.

In [ ]:
# ================================================================
# REPLACE ALL QUESTIONS AND GROUND TRUTHS with content from your PDF
# These are placeholder examples — your answers will be different
# ================================================================

GOLDEN_DATASET = [
    {
        "question": "What is the main topic of this document?",
        "ground_truth": "Write the actual answer from your PDF here"
    },
    {
        "question": "What methodology does the paper use?",
        "ground_truth": "Write the actual answer from your PDF here"
    },
    {
        "question": "What are the key findings?",
        "ground_truth": "Write the actual answer from your PDF here"
    },
    {
        "question": "Who are the authors of this paper?",
        "ground_truth": "Write the actual answer from your PDF here"
    },
    {
        "question": "What dataset was used in the experiments?",
        "ground_truth": "Write the actual answer from your PDF here"
    },
    # Add 15 more pairs from your PDF below:
    # {"question": "...", "ground_truth": "..."},
]

print(f"Golden dataset size: {len(GOLDEN_DATASET)} pairs")
print("Remember: replace placeholders with real Q&A from your PDF!")

## Step 3 — Run the pipeline on all questions

In [ ]:
print("Running pipeline on all questions...")
print("(This calls Groq API once per question — takes ~2 mins for 20 questions)\n")

eval_data = {
    "question":   [],
    "answer":     [],
    "contexts":   [],
    "ground_truth": []
}

for i, item in enumerate(GOLDEN_DATASET):
    q = item["question"]
    gt = item["ground_truth"]

    print(f"[{i+1}/{len(GOLDEN_DATASET)}] {q[:60]}...")

    try:
        answer, contexts = full_rag_pipeline(q)
        eval_data["question"].append(q)
        eval_data["answer"].append(answer)
        eval_data["contexts"].append(contexts)
        eval_data["ground_truth"].append(gt)
    except Exception as e:
        print(f"  Error: {e} — skipping")

print(f"\nDone. {len(eval_data['question'])} questions processed.")

## Step 4 — Run Ragas evaluation

In [ ]:
dataset = Dataset.from_dict(eval_data)

print("Running Ragas evaluation...")
print("(Uses Groq to evaluate — takes ~3 mins)\n")

results = evaluate(
    dataset=dataset,
    metrics=[
        faithfulness,       # are claims supported by retrieved chunks?
        answer_relevancy,   # does the answer address the question?
        context_precision,  # were the right chunks retrieved?
    ]
)

print("\n" + "="*50)
print(" RAGAS EVALUATION RESULTS")
print("="*50)
print(f" Faithfulness      : {results['faithfulness']:.4f}")
print(f" Answer Relevancy  : {results['answer_relevancy']:.4f}")
print(f" Context Precision : {results['context_precision']:.4f}")
avg = (results['faithfulness'] + results['answer_relevancy'] + results['context_precision']) / 3
print(f" Average Score     : {avg:.4f}")
print("="*50)
print("\nScore guide: > 0.7 = good | > 0.85 = excellent")

## Step 5 — Save results to disk

In [ ]:
os.makedirs("../data", exist_ok=True)

output = {
    "summary": {
        "faithfulness":      round(results['faithfulness'], 4),
        "answer_relevancy":  round(results['answer_relevancy'], 4),
        "context_precision": round(results['context_precision'], 4),
        "average":           round(avg, 4),
        "dataset_size":      len(GOLDEN_DATASET),
        "model":             "llama3-8b-8192",
        "embedding_model":   "all-MiniLM-L6-v2",
        "reranker":          "ms-marco-MiniLM-L-6-v2"
    },
    "per_question": [
        {
            "question": eval_data["question"][i],
            "answer":   eval_data["answer"][i],
            "ground_truth": eval_data["ground_truth"][i]
        }
        for i in range(len(eval_data["question"]))
    ]
}

with open("../data/ragas_results.json", "w") as f:
    json.dump(output, f, indent=2)

print("Results saved to ../data/ragas_results.json")
print("\nUpdate these scores in your README and LinkedIn post!")

## Key observations
- **Faithfulness < 0.5**: your prompt is too loose — tighten the system prompt
- **Answer Relevancy < 0.5**: retrieved chunks are off — try better chunking strategy
- **Context Precision < 0.5**: retrieval is noisy — tune hybrid search weights
- Screenshot the results table — this is the metric you quote in interviews

**In an interview say:** *'I evaluated DocMind using Ragas on a 20-question golden dataset and achieved a faithfulness score of X.XX, meaning X% of claims in answers are directly supported by retrieved text.'*

**Congratulations — you have built a production RAG system.**